# 01 · Setup and look at the data
Sets up Colab, downloads the training dataset to Google Drive, and **looks at it before we train anything**:
how many rows are really prescriptions, what the labels look like, and whether the same prescription
appears in more than one split.

**No GPU needed for this notebook** (leave the runtime on CPU to save your free GPU time).

> Reminder: this project is an assistive tool, not a medical decision-maker.
> Every output must be checked by a human against the original prescription.

### Setup
Mounts Google Drive (where data, checkpoints and results are kept), pulls the latest code from GitHub
and installs the extra packages. Run it every time you open the notebook.

**Private repo?** Click the key icon in Colab's left sidebar (Secrets), add a secret named `GITHUB_TOKEN`
holding a GitHub personal access token, and turn on notebook access.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/AlstonPR/PrescriptionScanner-ml.git"
REPO_DIR = "/content/PrescriptionScanner-ml"

try:  # use the token for a private repo; public repos work without it
    from google.colab import userdata
    REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none (fine for this notebook; Runtime > Change runtime type > T4 GPU for training)")

## Step 1 · Download the dataset to Drive
`naazimsnh02/medocr-vision-dataset` is about 0.5 GB. It is cached on Drive, so later runs don't download again.
If the download fails, the cell tells you what to try.

In [ ]:
from datasets import load_dataset
from src import config

config.HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
try:
    ds = load_dataset(config.DATASET_ID, cache_dir=str(config.HF_CACHE_DIR))
except Exception as e:
    print("Download failed:", e)
    print("Try: re-run this cell (network blips are common), or Runtime > Disconnect and delete runtime, then run all again.")
    raise
print(ds)

## Step 2 · What columns and splits does it have?
The dataset card says there are only two columns, `image` and `text`. Check that this is true.

In [ ]:
for name, part in ds.items():
    print(f"{name:12} {len(part):5d} rows | columns: {part.column_names}")

## Step 3 · Which rows are prescriptions?
The dataset mixes prescriptions with lab reports, invoices and forms, and has no column that says which is which.
We recognise prescriptions by their label format (`doctor_name: ... medications: ...`).
**Check this count against the pictures in Step 4**, because the rule is a guess about the label format.

In [ ]:
import pandas as pd
from src import medocr_data

idx = medocr_data.prescription_indices(ds)
summary = pd.DataFrame({"all rows": {s: len(ds[s]) for s in ds},
                        "prescriptions": {s: len(idx[s]) for s in ds}})
summary["other (dropped)"] = summary["all rows"] - summary["prescriptions"]
summary.loc["total"] = summary.sum()
display(summary)

## Step 4 · Look at some prescriptions and their labels
Do the labels match what you can read in the pictures? Are the pictures handwritten, printed, or fake-looking templates?
(Only prescription rows are shown. Lab reports are skipped because some contain patient names.)

In [ ]:
import matplotlib.pyplot as plt

rows = [(s, i) for s in ("train",) for i in idx[s][:4]]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (s, i) in zip(axes.flat, rows):
    ax.imshow(ds[s][i]["image"].convert("RGB")); ax.axis("off"); ax.set_title(f"{s} #{i}")
plt.tight_layout(); plt.show()

for s, i in rows:
    print(f"--- {s} #{i} ---")
    print(ds[s][i]["text"][:600], "\n")

## Step 5 · Do labels repeat? (templates, near-copies)
The dataset card hints the prescription labels may be very uniform. Count how many distinct doctor, clinic
and patient names there are. If a few names repeat everywhere, the data is probably synthetic and a model
can learn the template instead of reading handwriting. That belongs in the report.

In [ ]:
import re

def field(text, key):
    m = re.search(rf"{key}:\s*(.+?)(?=\s+[a-z_]+:\s|$)", text)
    return m.group(1).strip() if m else None

pres = pd.DataFrame([{"split": s, "text": ds[s][i]["text"]} for s in ds for i in idx[s]])
for key in ("doctor_name", "clinic_name", "patient_name", "date"):
    pres[key] = pres["text"].map(lambda t: field(t, key))
    print(f"{key:14} {pres[key].nunique():4d} distinct values over {len(pres)} prescriptions; "
          f"most common: {pres[key].value_counts().head(3).to_dict()}")

## Step 6 · Leakage check across splits
Exact duplicate images, or identical labels, in two different splits would make test scores look better than they are.
(Takes a minute or two: it opens every prescription image once.)

In [ ]:
report = medocr_data.leakage_report(ds, idx)
display(pd.DataFrame(report).T.rename_axis(["split A", "split B"]))
print("Anything above 0 means the same prescription (or a copy) is in two splits.")

## ✅ Check before moving on
- Step 2: columns are exactly `image` and `text`. If not, tell me what they are.
- Step 3: the number of prescriptions is near 1,000 in total (the card says about 1,000). If it's 0 or tiny, our
  prescription rule doesn't match the real label format, so send me the output of Step 4.
- Step 4: the labels match the pictures. Note whether the images look handwritten or like clean templates.
- Step 5: if only a few distinct doctor, clinic and patient names repeat, the data is probably synthetic. Tell me the numbers.
- Step 6: both columns are 0 for every pair. If not, send me the table and we fix the split before training.

Send me the printed output of Steps 3, 5 and 6. I'll write the cleaning, split and training code from the real numbers.
Next: **02_baseline_and_eda.ipynb**